In [1]:
%pip install pandas numpy pyarrow

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path

In [3]:
PROJECT_DIR = Path.cwd()

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = RAW_DIR / "PerfectPetClub.db"
CRM_PATH = RAW_DIR / "PerfectPetClub_CRM_report.csv"
SERVER_PATH = RAW_DIR / "PerfectPetClub_server_log.csv"
SURVEY_PATH = RAW_DIR / "PerfectPetClub_survey.csv"

In [4]:
def extract_data():
    conn = sqlite3.connect(DB_PATH)

    tables = [
        "customer", "payment", "product",
        "product_category", "product_review",
        "sale", "sale_product", "subscription",
        "subscription_type"
    ]

    data = {
        table: pd.read_sql_query(
            f"SELECT * FROM {table}", conn
        )
        for table in tables
    }

    conn.close()

    data["crm"] = pd.read_csv(CRM_PATH)
    data["server"] = pd.read_csv(SERVER_PATH)
    data["survey"] = pd.read_csv(SURVEY_PATH)

    return data

In [5]:
data = extract_data()

for name, df in data.items():
    print(name, df.shape)

customer (221, 9)
payment (23188, 4)
product (194, 4)
product_category (26, 2)
product_review (10990, 5)
sale (23188, 6)
sale_product (59759, 4)
subscription (221, 5)
subscription_type (3, 3)
crm (7585, 4)
server (67249, 5)
survey (221, 5)


In [7]:
def transform_data(data):

    customers = data["customer"].drop_duplicates()
    sales = data["sale"].drop_duplicates()
    sale_products = data["sale_product"].drop_duplicates()
    reviews = data["product_review"].drop_duplicates()
    crm = data["crm"].drop_duplicates()
    server = data["server"].drop_duplicates()
    subscriptions = data["subscription"].drop_duplicates()
    survey = data["survey"].drop_duplicates()

    sales["order_date"] = pd.to_datetime(
        sales["order_date"], errors="coerce"
    )

    # Sales features
    sales_features = sales.groupby("customer_id").agg(
        total_orders=("id", "count"),
        last_order_date=("order_date", "max")
    ).reset_index()

    sales_with_amount = sales.merge(
        data["payment"][["id", "amount"]],
        left_on="payment_id",
        right_on="id",
        how="left"
    )

    spend_features = sales_with_amount.groupby(
        "customer_id"
    ).agg(
        total_spend=("amount", "sum"),
        average_order_value=("amount", "mean")
    ).reset_index()

    sales_features = sales_features.merge(
        spend_features, on="customer_id", how="left"
    )

    # Item features
    item_data = sale_products.merge(
        sales[["id", "customer_id"]],
        left_on="sale_id",
        right_on="id",
        how="left"
    )

    item_features = item_data.groupby(
        "customer_id"
    ).agg(
        total_items=("quantity", "sum")
    ).reset_index()

    # Review features
    review_features = reviews.groupby(
        "customer_id"
    ).agg(
        average_rating=("star_rating", "mean"),
        total_reviews=("id", "count")
    ).reset_index()

    # CRM features
    crm_features = crm.groupby(
        "customer_id"
    ).size().reset_index(name="total_crm_interactions")

    # Server login features
    server_features = server.groupby(
        "login"
    ).size().reset_index(name="total_logins")

    server_features = server_features.merge(
        customers[["id", "email"]],
        left_on="login",
        right_on="email",
        how="inner"
    )[["id", "total_logins"]].rename(
        columns={"id": "customer_id"}
    )

    # Subscription information
    subscription_features = customers[
        ["id", "subscription_id"]
    ].merge(
        subscriptions[["id", "auto_renew", "subscription_type_id"]],
        left_on="subscription_id",
        right_on="id",
        how="left"
    )[["id_x", "auto_renew", "subscription_type_id"]]

    subscription_features = subscription_features.rename(
        columns={"id_x": "customer_id"}
    )

    # Combine customer-level features
    final = customers[["id", "email"]].rename(
        columns={"id": "customer_id"}
    )

    feature_tables = [
        sales_features,
        item_features,
        review_features,
        crm_features,
        server_features,
        subscription_features
    ]

    for table in feature_tables:
        final = final.merge(
            table, on="customer_id", how="left"
        )

    # Join survey using email
    final = final.merge(
        survey, on="email", how="left"
    )

    # Churn target
    final["churn"] = (
        final["auto_renew"] == 0
    ).astype(int)

    # Remove target leakage and identifiers
    final = final.drop(
        columns=["auto_renew", "email", "customer_id"],
        errors="ignore"
    )

    # Feature engineering
    reference_date = final["last_order_date"].max()

    final["days_since_last_order"] = (
        reference_date - final["last_order_date"]
    ).dt.days

    final = final.drop(columns=["last_order_date"])

    # Encode categorical columns
    final = pd.get_dummies(
        final,
        columns=["reported_income"],
        dtype=int
    )

    # Handle missing values
    for column in final.columns:
        if column != "churn":
            final[column] = final[column].fillna(
                final[column].median()
            )

    return final

In [8]:
final_data = transform_data(data)

print(final_data.shape)
final_data.head()

(221, 34)


,total_orders,total_spend,average_order_value,total_items,average_rating,total_reviews,total_crm_interactions,total_logins,subscription_type_id,nps_recommend,...,reported_income_46k-50k,reported_income_50k-54k,reported_income_54k-58k,reported_income_58k-62k,reported_income_62k-66k,reported_income_66k-70k,reported_income_70k-74k,reported_income_74k-78k,reported_income_78k-82k,reported_income_98k-102k
0,20,74455,3722.750000,46,2.400000,10.0,3.0,48,3,8,...,0,0,0,0,0,0,0,0,0,0
1,253,1036961,4098.660079,748,3.176991,113.0,45.0,1085,3,5,...,0,0,0,0,0,0,0,0,0,0
2,304,1284692,4225.960526,824,3.082090,134.0,167.0,624,1,9,...,0,0,0,0,0,0,0,0,0,0
3,1055,9577504,9078.202844,7362,3.299814,537.0,281.0,1540,1,9,...,0,0,0,0,0,0,0,0,0,0
4,44,179738,4084.954545,116,3.555556,18.0,5.0,388,3,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
print("Missing values:")
print(final_data.isnull().sum().sum())

print("Duplicate rows:", final_data.duplicated().sum())

print("Churn distribution:")
print(final_data["churn"].value_counts())

Missing values:
0
Duplicate rows: 0
Churn distribution:
churn
0    135
1     86
Name: count, dtype: int64


In [10]:
def load_data(df):
    output_path = PROCESSED_DIR / "perfectpetclub_processed.parquet"
    df.to_parquet(output_path, index=False)
    print("Saved:", output_path)

load_data(final_data)

Saved: /home/7d93c896-af26-470c-860b-5614ac2ea659/PerfectPetClub/data/processed/perfectpetclub_processed.parquet


In [11]:
final_data = transform_data(data)

print("Dataset shape:", final_data.shape)

print("\nDays since last order by churn:")
print(
    final_data.groupby("churn")["days_since_last_order"]
    .describe()
)

Dataset shape: (221, 34)

Days since last order by churn:
       count        mean         std  min   25%    50%     75%    max
churn                                                                
0      135.0    2.022222    2.691797  0.0   0.0    1.0    3.00   12.0
1       86.0  149.895349  101.455876  2.0  59.0  133.5  242.75  313.0


In [12]:
import inspect

print(inspect.getsource(transform_data))

def transform_data(data):

    customers = data["customer"].drop_duplicates()
    sales = data["sale"].drop_duplicates()
    sale_products = data["sale_product"].drop_duplicates()
    reviews = data["product_review"].drop_duplicates()
    crm = data["crm"].drop_duplicates()
    server = data["server"].drop_duplicates()
    subscriptions = data["subscription"].drop_duplicates()
    survey = data["survey"].drop_duplicates()

    sales["order_date"] = pd.to_datetime(
        sales["order_date"], errors="coerce"
    )

    # Sales features
    sales_features = sales.groupby("customer_id").agg(
        total_orders=("id", "count"),
        last_order_date=("order_date", "max")
    ).reset_index()

    sales_with_amount = sales.merge(
        data["payment"][["id", "amount"]],
        left_on="payment_id",
        right_on="id",
        how="left"
    )

    spend_features = sales_with_amount.groupby(
        "customer_id"
    ).agg(
        total_spend=("amount", "sum"),
        average_ord

In [13]:
print("Missing last order dates:")
print(final_data["days_since_last_order"].isna().sum())

print("\nChurn distribution by days since last order:")
print(
    final_data.groupby("churn")["days_since_last_order"]
    .agg(["min", "max", "mean", "median"])
)

Missing last order dates:
0

Churn distribution by days since last order:
       min  max        mean  median
churn                              
0        0   12    2.022222     1.0
1        2  313  149.895349   133.5


In [14]:
print("Churn labels:")
print(final_data["churn"].value_counts())

print("\nAuto-renew relationship:")
print(
    data["subscription"]["auto_renew"].value_counts(
        dropna=False
    )
)

Churn labels:
churn
0    135
1     86
Name: count, dtype: int64

Auto-renew relationship:
auto_renew
1    135
0     86
Name: count, dtype: int64
